### Import Library

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report
from sklearn.model_selection import GridSearchCV
from xgboost import XGBClassifier
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import LabelEncoder

import pickle as pkl
import warnings
warnings.filterwarnings("ignore")

Error: No connection selected.

### Load the Dataset

In [2]:
df = pd.read_csv("data_C.csv")

Error: No connection selected.

# Data Preprocessing

In [3]:
df.head()

Error: No connection selected.

In [4]:
df.tail()

Error: No connection selected.

In [5]:
df.shape

Error: No connection selected.

The dataset consists of 41,258 samples (rows) and 15 features (columns).

In [6]:
df.columns

Error: No connection selected.

In [7]:
df.info()

Error: No connection selected.

In [8]:
df.isna().sum()

Error: No connection selected.

There are several missing values in the `CreditScore` column, with a relatively small number of 12 entries. Given that the proportion of missing values is low relative to the overall dataset, the most appropriate option is to remove the rows containing those missing values.

This removal process is carried out because its impact on the overall integrity of the data is minimal. By removing entries with missing values, we can maintain high data quality without sacrificing the significance of the dataset.

In [9]:
df.dropna(subset=['CreditScore'], inplace=True)
df.isna().sum()

Error: No connection selected.

The dataset is clean and contains no missing values.

In [10]:
print('Number of Duplicated Data:', df.duplicated().sum())

Error: No connection selected.

The dataset is also free of duplicate values. Next, I will analyze the first three columns of this dataset.

In [11]:
df['Unnamed: 0'].value_counts()

Error: No connection selected.

In [12]:
df['id'].value_counts()

Error: No connection selected.

In [13]:
df['CustomerId'].value_counts()

Error: No connection selected.

The three columns analyzed above—`Unnamed: 0`, `id`, and `CustomerId`—do not provide relevant or useful information for the data analysis.

Therefore, I will drop these three columns for the following reasons:
1. `Unnamed: 0`: This column appears to be an index or row number generated when saving or loading data in a file format. It does not contribute anything to understanding or analyzing the data, as it only represents automatically generated row identification. In the context of data analysis, this is considered irrelevant metadata and can be ignored.
2. `id`: This column also appears to be a unique identifier for each entity in the dataset, which is often necessary in database systems but not required for statistical analysis or modeling. It does not provide any insight into the attributes or behaviors we want to study. Therefore, this column is considered irrelevant and can be removed without losing important information.
3. `CustomerId`: Similar to the `id` column, `CustomerId` contains a unique identifier for each customer in the dataset. Although this may be important in a database management context or a business application, it does not provide additional insight into customer behavior or characteristics relevant to the analysis. For this reason, it is also considered irrelevant and can be removed to simplify the dataset.

In [14]:
df.drop(columns = ['Unnamed: 0', 'id', 'CustomerId'], inplace = True)
df.head()

Error: No connection selected.

In [15]:
df[df['churn'] == 1]

Error: No connection selected.

The three ID and index columns have been removed, and the dataset is now simpler.

In [16]:
df['Surname'].value_counts()

Error: No connection selected.

Since the main task is to build a customer churn classification model, the `Surname` column can be removed because it does not contribute significantly to the churn prediction objective, for the following reasons:

1. **Not correlated with churn**: A customer's last name or surname has no direct correlation with the decision to churn or stay. Factors such as customer experience with the bank, satisfaction, or financial needs are more likely to influence churn decisions.

2. **Does not provide useful information**: In the context of churn prediction, more relevant information includes customer behavior and characteristics such as transaction history, account balance, account activity, and other indicators. A surname does not provide additional insight into customer behavior or tendencies that can be used in building a churn model.

In [17]:
df.drop(columns = ['Surname'], inplace = True)
df.head()

Error: No connection selected.

In [18]:
for column in df.columns:
    unique = df[column].value_counts()
    print("Column", column, ":", unique)
    print('')

Error: No connection selected.

# Exploratory Data Analysis

In [19]:
categorical_columns = ['Geography', 'Gender', 'NumOfProducts', 'HasCrCard', 'IsActiveMember', 'Tenure','churn']
numeric_columns = ['CreditScore', 'Age', 'Balance', 'EstimatedSalary']
print("Categorical columns:", categorical_columns)
print("Numeric columns:", numeric_columns)

Error: No connection selected.

I separated the numerical and categorical columns to make the analysis process easier.

### Numerical

**Descriptive Statistics**

In [20]:
From the descriptive statistics above, I gained several insights:

1. **CreditScore**:
   - The average credit score of customers is approximately 655.8.
   - Credit scores range widely from 350 to 850.
   - The relatively low standard deviation indicates that most credit scores are not far from the mean.

2. **Age**:
   - The average customer age is approximately 38 years.
   - Customer ages range from 18 to 92 years.
   - The age distribution tends to lean toward younger customers, with the 25th percentile at 32 years.

3. **Balance**:
   - The average account balance is approximately $55,325.
   - Many customers have a zero account balance, as the median and the 25th percentile are both 0.
   - The fairly large standard deviation indicates significant variation in account balances.

4. **EstimatedSalary**:
   - The average estimated salary is approximately $112,501.
   - Estimated salaries range from $11.58 to $199,992.48.
   - The distribution of salary is more evenly spread than that of account balance, with the median closer to the mean.

Error: No connection selected.

From the descriptive statistics above, I obtained several insights:

1. **CreditScore**:
   - The average customer credit score is approximately 655.8.
   - Credit scores have a fairly wide range from 350 to 850.
   - The relatively low standard deviation indicates that most credit scores are not far from the average.

2. **Age**:
   - The average customer age is approximately 38 years.
   - Customer ages vary from 18 to 92 years.
   - The age distribution tends to skew slightly toward younger customers, with the 25th percentile (25%) at 32 years.

3. **Balance**:
   - The average customer account balance is approximately $55,325.
   - A large portion of customers have a zero account balance (0), since the median and the 25th percentile are both 0.
   - The relatively large standard deviation indicates significant variation in account balances.

4. **EstimatedSalary**:
   - The average estimated salary is approximately $112,501.
   - Predicted salaries range from $11.58 to $199,992.48.
   - The salary distribution is more evenly spread than the account balance distribution, with the median closer to the average.

**Data Distribution using Histogram**

In [21]:
sns.set(style="whitegrid")
plt.figure(figsize=(12, 10))

plt.subplot(2, 2, 1)
sns.histplot(data=df, x='CreditScore', kde=True, color='skyblue', bins=30)
plt.title('Distribution of Credit Score')

plt.subplot(2, 2, 2)
sns.histplot(data=df, x='Age', kde=True, color='salmon', bins=30)
plt.title('Distribution of Age')

plt.subplot(2, 2, 3)
sns.histplot(data=df, x='Balance', kde=True, color='green', bins=30)
plt.title('Distribution of Balance')

plt.subplot(2, 2, 4)
sns.histplot(data=df, x='EstimatedSalary', kde=True, color='orange', bins=30)
plt.title('Distribution of Estimated Salary')

plt.tight_layout()
plt.show()

Error: No connection selected.

Observations about the varying data distributions across different columns can provide valuable insights into the characteristics of the dataset. Here are some key insights derived from the differences in the data distribution for the `Balance` and `EstimatedSalary` columns:

1. `Balance` (Account Balance):
   - The data distribution is right-skewed, indicating that many customers have low or even zero account balances.
   - This suggests that a large portion of customers may not keep much money in their accounts or may not use their bank accounts actively.
   - It is likely that some customers are not using financial services intensively or may be using services unrelated to savings or investment.

2. `EstimatedSalary` (Estimated Salary):
   - The data distribution is left-skewed, indicating that many customers have lower salaries.
   - This may reflect economic inequality among customers, where a large portion earn relatively low wages.
   - Significant differences in estimated salary can influence financial behavior and customer decisions, such as saving habits, use of financial products, or the likelihood of churn.

3. `Age` (Age):
   - The distribution is fairly symmetric and resembles a normal bell curve, indicating that most customers fall within a fairly uniform age range.
   - Younger customers are more common than older ones, which is reflected in the median and 25th percentile values being below the mean age.
   - Further analysis can help determine the effect of age on customer financial behavior, such as preferences for certain products, churn risk, or credit risk.

4. `CreditScore` (Credit Score):
   - The distribution also follows a mostly symmetric and bell-shaped pattern, indicating that most customers have varied but relatively similar credit scores.
   - The relatively high average credit score and low standard deviation suggest that most customers have good credit standing, with limited variation.
   - Higher credit scores may indicate stronger credit quality, which can affect access to financial products and services and influence churn risk.

**Check Outliers using Boxplot**

In [22]:
sns.set(style="whitegrid")
plt.figure(figsize=(12, 10))

plt.subplot(2, 2, 1)
sns.boxplot(data=df, x='CreditScore', color='skyblue')
plt.title('Distribution of Credit Score')

plt.subplot(2, 2, 2)
sns.boxplot(data=df, x='Age', color='salmon')
plt.title('Distribution of Age')

plt.subplot(2, 2, 3)
sns.boxplot(data=df, x='Balance', color='green')
plt.title('Distribution of Balance')

plt.subplot(2, 2, 4)
sns.boxplot(data=df, x='EstimatedSalary', color='orange')
plt.title('Distribution of Estimated Salary')

plt.tight_layout()
plt.show()


Error: No connection selected.

Credit scores and age may have outliers due to natural variation in the population. In the case of credit scores, some individuals may have very low or very high scores because of poor or excellent credit history. For age, outliers may occur because of variation in the age distribution of the population, such as very young or very old individuals. Outliers in both cases may represent real but uncommon situations, or they may also be due to errors in data collection.

### Categorical

In [23]:
plt.figure(figsize=(15, 12))

for i, column in enumerate(categorical_columns, 1):
    plt.subplot(3, 3, i)
    plt.title(column)
    plt.pie(df[column].value_counts(), labels=df[column].value_counts().index, autopct='%1.1f%%', startangle=140)
    plt.axis('equal')  

plt.subplots_adjust(wspace=0.7, hspace=0.7)
plt.tight_layout()
plt.show()

Error: No connection selected.

In [24]:
plt.figure(figsize=(15, 12))

for i, column in enumerate(categorical_columns, 1):
    plt.subplot(3, 3, i)
    plt.title(column)
    df[column].value_counts().plot(kind='bar', color='skyblue')
    plt.ylabel('Frequency')

plt.subplots_adjust(wspace=0.5, hspace=0.5)

plt.tight_layout()
plt.show()

Error: No connection selected.

Based on the EDA results, I can conclude several important findings about the characteristics of customers in the dataset:

1. **Geography**:
   - The frequency distribution of customers by country shows that most customers come from France, followed by Spain and Germany.
   - Although France has the highest number of customers, the difference between the customer counts in Spain and Germany is relatively small, suggesting that the bank has a significant market share in both countries.

2. **Gender**:
   - The analysis shows that there are more male customers than female customers in the dataset.
   - This information can help the bank develop more focused marketing strategies and account for different preferences or needs between the two genders.

3. **NumOfProducts**:
   - The frequency distribution of the number of products owned by customers shows that most customers have 1 or 2 products, while relatively few have 3 or 4 products.
   - This may indicate certain product preferences among customers, and the bank can use this information to optimize product offerings or tailor marketing strategies.

4. **HasCrCard**:
   - The analysis shows that most customers have a credit card, while relatively few do not.
   - This suggests that credit card services may be a popular product among customers, and the bank can consider strengthening or expanding these services further.

5. **IsActiveMember**:
   - The distribution of active members shows that most customers are active members, although the number of inactive customers is also significant.
   - This is important because active customers tend to be more engaged and more likely to maintain their relationship with the bank. The bank can take steps to increase activity among inactive customers.

6. **Tenure**:
   - The customer tenure data shows significant variation, with a range of 7 months being the highest, followed by 2 months and 8 months.
   - This indicates that some customers have been with the bank for a long time, while others are relatively new or have shorter relationships.

7. **Churn**:
   - There is a class imbalance in the churn target, where most customers do not churn (0), while about 21% do churn (1).
   - This class imbalance can lead to an imbalanced model and biased evaluation. Therefore, special measures such as oversampling or using appropriate evaluation metrics are needed in modeling to minimize its impact.

# Split Data into Train and Test Data

Splitting the data before feature engineering is essential to prevent information leakage from the test set into the model training process. This ensures a clear separation between the data used for model development (the training set) and the unseen data used for evaluation (the test set). By performing feature engineering only on the training set, we can make sure the model learns from genuine patterns in the data without using information that should not be accessible to it. This preserves the integrity of the evaluation process and allows for a more accurate assessment of the model's performance on previously unseen data.

In [25]:
input = df.drop('churn', axis = 1)
output = df['churn']

x_train, x_test, y_train, y_test = train_test_split(input, output, test_size = 0.2, random_state = 42)

Error: No connection selected.

In [26]:
print('x_train: ', x_train.shape)
print('x_test: ', x_test.shape)
print('y_train: ', y_train.shape)
print('y_test: ', y_test.shape)

Error: No connection selected.

# Feature Engineering

### Feature Encoding (Binary Encoding)

For categorical variables such as `Gender` (where there are only two possible categories, such as Male and Female), using binary encoding can be an efficient choice. In binary encoding, a single new column is sufficient to represent the variable, where the categories are encoded as binary values (0 or 1). For example, using binary encoding, we could represent Male as 0 and Female as 1, or vice versa.

In [27]:
BinaryEncode = {"Gender": {"Male" : 1,"Female" :0}}
x_train = x_train.replace(BinaryEncode)
x_test = x_test.replace(BinaryEncode)   

Error: No connection selected.

In [28]:
x_train.head()

Error: No connection selected.

In [29]:
x_test.head()

Error: No connection selected.

In [ ]:
filename = 'gender_encoder.pkl'
pkl.dump(BinaryEncode, open(filename, 'wb'))

In [ ]:
HasCrCardEncode = {"HasCrCard": {"Yes":1,"No" :0}}
filename = 'hasCrCard_encoder.pkl'
pkl.dump(HasCrCardEncode, open(filename, 'wb'))

In [ ]:
IsActiveMemberEncode = {"IsActiveMember": {"Yes":1,"No" :0}}
filename = 'isActiveMember_encoder.pkl'
pkl.dump(IsActiveMemberEncode, open(filename, 'wb'))

### Feature Encoding (One Hot Encoding)

One-hot encoding is a technique used in data preprocessing to convert categorical variables into a format that is more suitable for machine learning models. When dealing with a categorical variable like `Geography` (country), where there is no meaningful order between the categories, it is not appropriate to treat it as a numeric variable directly because that would introduce ambiguity. Instead, one-hot encoding can be used.

In one-hot encoding, each unique value in the categorical variable is mapped to a new column. If there are three countries (Germany, France, Spain), three new columns will be created, each representing whether a customer belongs to that country. For example, if a customer is from France, the France column will have a value of 1 while the Germany and Spain columns will have a value of 0.

In [ ]:
geo_enc_train = x_train[['Geography']]
geo_enc_test = x_test[['Geography']]

train_encoded_geo = OneHotEncoder()

geo_enc_train = pd.DataFrame(train_encoded_geo.fit_transform(geo_enc_train).toarray(),columns=train_encoded_geo.get_feature_names_out())
geo_enc_test = pd.DataFrame(train_encoded_geo.transform(geo_enc_test).toarray(),columns=train_encoded_geo.get_feature_names_out())

x_train = x_train.reset_index()
x_test = x_test.reset_index()

x_train_enc = pd.concat([x_train,geo_enc_train], axis=1)
x_train_enc = x_train_enc.drop(['Geography'], axis=1)
x_test_enc = pd.concat([x_test,geo_enc_test], axis=1)
x_test_enc = x_test_enc.drop(['Geography'], axis=1)

In [ ]:
train_encoded_geo.get_feature_names_out()

array(['Geography_France', 'Geography_Germany', 'Geography_Spain'],
      dtype=object)

### Feature Scaling (Standard Scaler)

In [ ]:
ColScale = ['CreditScore', 'Age', 'Balance', 'EstimatedSalary']
Scaler = StandardScaler()
x_train_enc[ColScale] = Scaler.fit_transform(x_train_enc[ColScale])
x_test_enc[ColScale] = Scaler.transform(x_test_enc[ColScale])

In [ ]:
filename_scale = 'standard_scaler.pkl'
pkl.dump(Scaler, open(filename_scale, 'wb'))

In [ ]:
x_train_enc.head()

,index,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Geography_France,Geography_Germany,Geography_Spain
0,41120,-0.224995,0,-0.128407,0,-0.880551,2,1,0,-0.291066,1.0,0.0,0.0
1,6084,0.226278,1,0.653142,3,0.536656,1,1,1,-0.770330,0.0,1.0,0.0
2,8104,-0.237530,1,-1.133257,5,1.171809,2,1,0,1.372995,0.0,1.0,0.0
3,23634,-1.854593,1,3.667691,9,-0.880551,2,1,1,-1.467377,1.0,0.0,0.0
4,23943,-0.036964,1,-0.128407,3,-0.880551,2,1,1,-0.748091,1.0,0.0,0.0


In [ ]:
x_test_enc.head()

,index,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Geography_France,Geography_Germany,Geography_Spain
0,24999,-0.049500,0,-1.133257,4,0.940042,1,0,0,0.963162,0.0,1.0,0.0
1,8730,-0.713875,1,1.211392,10,-0.880551,1,1,0,0.980093,1.0,0.0,0.0
2,3583,-0.212460,1,-0.351707,9,0.810306,1,1,1,1.087561,0.0,1.0,0.0
3,35209,0.251349,1,0.206542,2,0.848709,2,0,1,0.452553,0.0,1.0,0.0
4,29762,-2.293332,0,0.541492,1,0.406924,1,1,0,1.734971,1.0,0.0,0.0


# Modelling

### 1.) Random Forest Classifier

In [ ]:
rf_classifier = RandomForestClassifier(criterion = 'gini', max_depth = 4, random_state= 42)
rf_classifier.fit(x_train_enc, y_train)

RandomForestClassifier(max_depth=4, random_state=42)

In [ ]:
y_predict = rf_classifier.predict(x_test_enc)
print(classification_report(y_test, y_predict))

              precision    recall  f1-score   support

           0       0.84      0.98      0.90      6471
           1       0.84      0.30      0.44      1779

    accuracy                           0.84      8250
   macro avg       0.84      0.64      0.67      8250
weighted avg       0.84      0.84      0.80      8250



**Hyperparameter Tuning (GridSearchCV)**

In [ ]:
param_grid = {'n_estimators': [50, 100],
              'criterion': ['gini', 'entropy'],
              'max_depth': [None, 10]}
            #   'min_samples_split': [2,5, 10],
            #   'min_samples_leaf': [1, 2, 4],
            #   'max_leaf_nodes': [None, 5, 10, 20]}

In [ ]:
rf_classifier_tuned = RandomForestClassifier(random_state = 42)
rf_classifier_tuned = GridSearchCV(rf_classifier_tuned,
                                   param_grid = param_grid,
                                   scoring = 'accuracy',
                                   cv = 5)

In [ ]:
rf_classifier_tuned.fit(x_train_enc,y_train)
print("Tuned Hyperparameters: ", rf_classifier_tuned.best_params_)
print("Accuracy: ", rf_classifier_tuned.best_score_)

Tuned Hyperparameters:  {'criterion': 'gini', 'max_depth': 10, 'n_estimators': 100}
Accuracy:  0.8613772288730616


### 2.) XGBoost Classifier

In [ ]:
xgb_classifier = XGBClassifier(random_state=42)
xgb_classifier.fit(x_train_enc, y_train)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric=None, feature_types=None,
              gamma=None, grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=None, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=None, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=None, n_jobs=None,
              num_parallel_tree=None, random_state=42, ...)

In [ ]:
y_predict_xgb = xgb_classifier.predict(x_test_enc)
print(classification_report(y_test, y_predict_xgb))

              precision    recall  f1-score   support

           0       0.89      0.94      0.91      6471
           1       0.72      0.56      0.63      1779

    accuracy                           0.86      8250
   macro avg       0.80      0.75      0.77      8250
weighted avg       0.85      0.86      0.85      8250



In [ ]:
param_grid_xgb = {'n_estimators': [50, 100],
                  'max_depth': [3, 5],
                  'learning_rate': [0.1, 0.01],
                  'subsample': [0.5, 0.7]
                  }

In [ ]:
xgb_classifier_tuned = XGBClassifier(random_state=42)
xgb_classifier_tuned = GridSearchCV(xgb_classifier_tuned,
                                    param_grid=param_grid_xgb,
                                    scoring='accuracy',
                                    cv=5)

In [ ]:
xgb_classifier_tuned.fit(x_train_enc, y_train)
print("Tuned Hyperparameters: ", xgb_classifier_tuned.best_params_)
print("Accuracy: ", xgb_classifier_tuned.best_score_)

Tuned Hyperparameters:  {'learning_rate': 0.1, 'max_depth': 3, 'n_estimators': 100, 'subsample': 0.5}
Accuracy:  0.8632258928120422


# Evaluation

In [ ]:
rf_classifier_best_tuned = rf_classifier_tuned.best_estimator_
y_predict = rf_classifier_best_tuned.predict(x_test_enc)
print(classification_report(y_test,y_predict))

              precision    recall  f1-score   support

           0       0.88      0.96      0.92      6471
           1       0.77      0.52      0.62      1779

    accuracy                           0.86      8250
   macro avg       0.83      0.74      0.77      8250
weighted avg       0.86      0.86      0.85      8250



In [ ]:
xgb_classifier_best_tuned = xgb_classifier_tuned.best_estimator_
y_predict_xgb_tuned = xgb_classifier_best_tuned.predict(x_test_enc)
print(classification_report(y_test, y_predict_xgb_tuned))

              precision    recall  f1-score   support

           0       0.88      0.95      0.92      6471
           1       0.76      0.55      0.64      1779

    accuracy                           0.86      8250
   macro avg       0.82      0.75      0.78      8250
weighted avg       0.86      0.86      0.86      8250



To determine the best model, it is necessary to consider several evaluation metrics, especially precision, recall, and F1-score for class 1 (churn), because we are interested in the model's ability to predict the minority class (churn).

From the two models provided:
**Model 1:**
- Precision for class 1: 0.77
- Recall for class 1: 0.52
- F1-score for class 1: 0.62

**Model 2:**
- Precision for class 1: 0.76
- Recall for class 1: 0.55
- F1-score for class 1: 0.64

If the focus is on model performance in predicting the minority class (churn), then the better model is **Model 2**. This is because Model 2 has a higher recall for class 1, indicating that it is better at identifying customers who truly churn. Although its precision is slightly lower, recall is more important in this context because we want to minimize false negatives (classifying customers who churn as non-churn).

# Save Best Model as Pickle

In [1]:
with open('xgb_classifier_model.pkl', 'wb') as file:
    pkl.dump(xgb_classifier_best_tuned, file)

Error: No connection selected.